In [ ]:
## Temporary solution
!pip install openai==1.55.3 httpx==0.27.2 --force-reinstall --quiet
import os
os.kill(os.getpid(), 9)

In [34]:
import json
from openai import AzureOpenAI
from tqdm.notebook import tqdm

client = AzureOpenAI(
  azure_endpoint="https://ner-generation-vnt.openai.azure.com/",
  api_key="c693434d2f0a40b69bdc8317737879fc",
  api_version="2024-02-01"
)

def run_prediction(input_text, model="gpt4-o-mini"):
    try:
        messages = [
          {
              "role": "system",
              "content": """
                Você é um especialista em redação clínica e anonimização de dados médicos.
                Sua tarefa é gerar prontuários médicos sintéticos em português brasileiro,
                seguindo rigorosamente padrões reais de registros hospitalares.
                Você deve respeitar integralmente as regras de anotação de entidades fornecidas pelo usuário.
      """
          },
          {
              "role": "user",
              "content": """Você receberá abaixo UM EXEMPLO DE PRONTUÁRIO MÉDICO JÁ ANOTADO.
Sua tarefa é gerar um novo prontuário médico sintético, obedecendo estritamente às seguintes regras:
Regras obrigatórias
- O texto gerado deve seguir o mesmo estilo clínico e estrutura narrativa do prontuário de exemplo (evolução médica, linguagem técnica, abreviações, etc.).
- O conteúdo deve ser realista, mas inteiramente sintético (não reutilize nomes, datas, locais ou números do exemplo).
- Você deve introduzir variabilidade no conteúdo (diagnósticos, especialidade, evolução, contexto clínico), desde que mantenha coerência médica.
- Cada prontuário gerado deve pertencer a uma especialidade médica potencialmente diferente (ex.: clínica médica, pediatria, oncologia, nefrologia, pneumologia, neurologia, cirurgia, ginecologia, psiquiatria). Evite repetir a mesma especialidade entre gerações consecutivas
- Escolha aleatoriamente uma especialidade médica antes de gerar o prontuário.
- Nenhum texto fora do prontuário médico deve ser gerado.
- Não inclua explicações, comentários ou cabeçalhos adicionais.
- Todas as entidades sensíveis devem estar explicitamente anotadas usando exatamente as mesmas tags do exemplo.
- Não crie novas categorias de entidades.
- Sempre que houver espaço entre palavras de uma mesma entidade, incluir apenas uma palavra por tag (ex: <HOSPITAL>Hospital</HOSPITAL/> <HOSPITAL>da</HOSPITAL/> <HOSPITAL>Amazonia</HOSPITAL/>)

As tags devem envolver apenas o texto da entidade, sem incluir palavras adjacentes. Entidades que DEVEM ser anotadas
<AGE> </AGE/>, <PHONE> </PHONE/>, <EMAIL> </EMAIL/>, <DATE> </DATE/>, <IDNUM> </IDNUM/>, <MEDICAL_RECORD> </MEDICAL_RECORD/>,
<HEALTH_PLAN> </HEALTH_PLAN/>, <STREET> </STREET/>, <CITY> </CITY/>, <ZIP> </ZIP/>, <STATE> </STATE/>, <COUNTRY> </COUNTRY/>,
<LOCATION_OTHER> </LOCATION_OTHER/>, <ORGANIZATION> </ORGANIZATION/>, <HOSPITAL> </HOSPITAL/>, <PATIENT> </PATIENT/>, <DOCTOR> </DOCTOR/>,
<PROFESSION> </PROFESSION/>, <OTHER> </OTHER/>

Importante:
- Use as mesmas convenções de anotação do exemplo (inclusive em entidades compostas).
- O resultado final deve conter somente o prontuário médico anotado.
- Escolha aleatoriamente uma especialidade médica antes de gerar o prontuário.

Exemplo de prontuário anotado: {}""".format(input_text)
          }
      ]

        response = client.chat.completions.create(
            model = model,
            messages = messages,
            temperature = 0.9,
            max_tokens = 4096,
            top_p = 0.9,
            frequency_penalty = 0,
            presence_penalty = 0,
            stop = None
        )
    except Exception as e:
        print(str(e))

    return response

In [19]:
train_data_path ='/content/clinical_deid_training_set_final.json'
test_data_path = '/content/clinical_deid_test_set_final.json'

# Open and read the test set
with open(train_data_path, 'r') as file:
    train_set = json.load(file)

# Open and read the test set
with open(test_data_path, 'r') as file:
    test_set = json.load(file)


train_set[0].keys()

#train_set_filtered = [ex['text'] for ex in train_set if ex['synthetic'] == False]
#test_set_filtered = [ex['text'] for ex in test_set if ex['synthetic'] == False]

#list_all_real = train_set_filtered + test_set_filtered

#len(train_set_filtered), len(test_set_filtered), len(list_all_real)

(874, 201, 1075)

In [ ]:
dataset_split = train_set # eval_set, test_set

list_syn = []
n_repetitions = 5
for input_text in tqdm(dataset_split):
    for i in range(n_repetitions):
        syn_text = run_prediction(input_text ,model="NER-generation-VNT").choices[0].message.content

        list_syn.append(syn_text)


In [ ]:
print(input_text[0:400])

for text_syn in list_syn:
    print(text_syn[0:400])
    print("")